STRUCTURED OUTPUT

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A4E92CB610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A4E9318190>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

PYDANTIC

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [6]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class Movie(BaseModel):
    title:str = Field(description="The name of the movie")
    year:int = Field(description="The year in which the movie was released")
    rating:float = Field(description="The rating of the movie")

agent_pydantic = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=Movie
)

agent_pydantic.invoke({"messages" : [{"role" : "user", "content" : "Tell me about the movie 50 Shades Of Grey"}]})

{'messages': [HumanMessage(content='Tell me about the movie 50 Shades Of Grey', additional_kwargs={}, response_metadata={}, id='ab5e01a8-e341-46c7-8050-4f6b73ea2fae'),
  AIMessage(content='{"title":"Fifty Shades of Grey","year":2015,"rating":6.2}', additional_kwargs={'reasoning_content': 'We need to produce JSON that conforms to the schema. The schema: object with properties rating (number), title (string), year (integer). Required fields: title, year, rating. Title of the schema is "Movie". So we need to output JSON object with those fields. We need to include rating, title, year. For "50 Shades of Grey", year 2015, rating? We might need rating maybe MPAA rating? It\'s a movie. Could use rating: PG-13? But rating is number type. Maybe rating is like IMDb rating? The schema says rating type number. So we can give rating e.g. 6.2? IMDb rating is 6.2. Let\'s use rating 6.2. Provide JSON with those fields. Ensure compact JSON: no extra spaces or newlines? It says compact JSON formatting. 

TYPEDDICT

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [7]:
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class Movie(TypedDict):
    title: Annotated[str,"The name of the movie"]
    year: Annotated[int,"The year in which the movie was released"]
    rating: Annotated[float,"The rating of the movie"]

agent_typeddict = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=Movie
)

agent_typeddict.invoke({"messages" : [{"role" : "user", "content" : "Tell me about the movie Inception"}]})

{'messages': [HumanMessage(content='Tell me about the movie Inception', additional_kwargs={}, response_metadata={}, id='92dac043-76e2-4692-b340-1a53b2b681b7'),
  AIMessage(content='{"title":"Inception","year":2010,"rating":8.8}', additional_kwargs={'reasoning_content': 'We need to output JSON with properties rating, title, year. Must include all required fields. Provide rating (number), title string, year integer. The user asked "Tell me about the movie Inception". We need to output a JSON object with those fields. Use compact JSON formatting. So something like:\n\n{"title":"Inception","year":2010,"rating":8.8} (rating is number). Ensure rating is number. Provide rating maybe 8.8. Use compact JSON. No extra keys.\n\nThus final.'}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 189, 'total_tokens': 327, 'completion_time': 0.15281196, 'completion_tokens_details': {'reasoning_tokens': 109}, 'prompt_time': 0.010694466, 'prompt_tokens_details': None, 'queue_ti

DATACLASSES

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [8]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Movie:
    title:str
    year:int
    rating:float

agent_dataclass = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=Movie
)

agent_dataclass.invoke({"messages" : [{"role" : "user", "content" : "Tell me about the movie Obsession"}]})

{'messages': [HumanMessage(content='Tell me about the movie Obsession', additional_kwargs={}, response_metadata={}, id='5ced18f8-3b91-40a1-9871-79b467351d82'),
  AIMessage(content='{"title":"Obsession","year":1976,"rating":7.2}', additional_kwargs={'reasoning_content': 'User wants info about movie "Obsession". We need to output JSON in the given schema. Must include all required fields: title, year, rating. Provide rating number. We need to output JSON only. Must be compact. Let\'s find info: "Obsession" could be multiple movies. Likely the 1976 film? There is "Obsession" 1976 film starring James Mason, etc. Year 1976. Rating? Maybe from IMDb rating 7.2? Let\'s check: IMDb rating for "Obsession" 1976 is 7.2. Alternatively "Obsession" (2021) Indian film? But likely 1976. Provide rating 7.2. Use number. Provide title exactly "Obsession". Year 1976. Output JSON. Ensure compact.'}, response_metadata={'token_usage': {'completion_tokens': 194, 'prompt_tokens': 189, 'total_tokens': 383, 'comp

NESTED STRUCTURE

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    age:int
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    director:str
    actors:list[Actor]
    rating:float

model_nested_structure = model.with_structured_output(MovieDetails, include_raw=True)
model_nested_structure.invoke("Tell me about the movie 50 Shades Of Grey")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Tell me about the movie 50 Shades Of Grey". We need to provide info about the movie. The system says we have a function called MovieDetails that can provide details. The user wants a summary. We can call the function with the appropriate parameters. We need to provide actors, director, rating, title, year. The function signature: MovieDetails expects an object with actors array (each with age, name, role), director string, rating number, title string, year number. We should provide that. Let\'s gather data: Title: 50 Shades of Grey. Year: 2015. Director: Sam Taylor-Johnson. Rating: maybe 7.0? But we can provide rating as 7.0 or 6.5. Actors: Dakota Johnson (age 31? Actually as of 2015, she was 29? Born 1989, so age 26 in 2015). But we can approximate. Christian Bale (age 43? Born 1974, age 41 in 2015). And others: Jamie Dornan (age 37? Born 1978, age 37 in 2015). Also others: Sam Taylor-Johnson. But w